In [1]:

import CPPMTO_optimizer1_tighter_flow_bound_iter_OD
from CPPMTO_optimizer1_tighter_flow_bound_iter_OD import solve_bilevel_model_with_tight_flow_bound_iter_OD
import CPPMTO_instance_parser_2
from CPPMTO_instance_parser_2 import parse_bilevel_instance1
import Json_solution_loader
from Json_solution_loader import load_solution_json
import os
import json
import pandas as pd
import random
import numpy as np

def iterative_tightening(json_file_path, save_folder, initial_Pbar, initial_M, ops, tol=1e-6, max_iter=20, verbose=True):
    """
    Iteratively tighten the OD-specific price bounds until the maximum relaxation gap is within tolerance.
    
    Parameters:
        json_file_path: str
            Path to the instance JSON file.
        save_folder: str
            Folder to save solution files.
        initial_Pbar: float
            Initial lower bound for the price.
        initial_M: float
            Initial upper bound for the price.
        tol: float
            Tolerance for |z - p*q|.
        max_iter: int
            Maximum number of iterations.
        verbose: bool
            Whether to print progress details.
    
    Returns:
        solution: dict (from the last call to solve_bilevel_model_with_tight_flow_bound)
        solution_dict: dict (detailed solution information)
        solution_file: str (path to the saved solution file)
    """
    # Use the parsed data:
    parsed_data = parse_bilevel_instance1(json_file_path)
    W = parsed_data["OD_pairs"]

    # Then initialize od_bounds:
    od_bounds = {(o, d): (initial_Pbar, initial_M) for (o, d) in W}

    # Initialize per-OD price bounds in a dictionary.
    
    iteration = 0
    solutions_list = []

    while iteration < max_iter:
        print(f"\nIteration {iteration}: Solving with OD-specific price bounds:")
        #for (o,d), bounds in od_bounds.items():
        #    print(f"  OD {o}->{d}: Price in [{bounds[0]:.3f}, {bounds[1]:.3f}]")
        
        # Call the solver with the current OD-specific bounds.
        # (Assume your solver is modified to accept a dictionary 'od_bounds' instead of scalar M and Pbar.)
        solution, solution_dict, solution_file = solve_bilevel_model_with_tight_flow_bound_iter_OD(
            json_file_path,
            save_folder,
            iteration,
            initial_M,
            initial_Pbar,
            od_bounds=od_bounds,  # pass OD-specific bounds
            ops=ops,
            verbose=verbose,
            verbose2=verbose
        )
        
        if solution_dict is None:
            print("No solution was found in this iteration. Exiting iterative process.")
            break

        # For each OD pair, gather bike prices from the selected PT+Bike arcs.
        new_od_bounds = {}
        for (o, d) in W:
            od_str = f"{o}->{d}"
            # Filter instances for this OD and type 'PT+Bike' with nu nearly 1.
            od_bike_prices = [entry.get("Bike Price", 0)
                              for entry in solution_dict.get("instances", [])
                              if entry.get("OD") == od_str and entry.get("Type") == "PT+Bike" and entry.get("nu", 0) > 0.5]
            if od_bike_prices:
                p_min = np.min(od_bike_prices)
                p_max = np.max(od_bike_prices)
            else:
                # If no bike prices found, keep the current bounds.
                p_min, p_max = od_bounds[(o, d)]
            # Optionally, you might want to narrow the bounds around the observed prices.
            # Here we update lower bound as max(initial_Pbar, observed minimum) and
            # update upper bound as min(initial_M, observed maximum minus a scaled gap).
            scale_upper = 0.05  # you may adjust this step-size factor as needed.
            new_lower = max(initial_Pbar, p_min)
            new_upper = min(initial_M, p_max - scale_upper * (p_max - p_min))
            new_od_bounds[(o, d)] = (new_lower, new_upper)
            print(f"  OD {o}->{d}: Representative prices: min={p_min:.3f}, max={p_max:.3f} -> New bounds: [{new_lower:.3f}, {new_upper:.3f}]")
        
        # Compute a global maximum relaxation gap for monitoring.
        relax_diffs = [abs(entry.get("Relaxation Difference", 0))
                       for entry in solution_dict.get("instances", [])
                       if entry.get("Type") == "PT+Bike"]
        if relax_diffs:
            global_gap = max(relax_diffs)
        else:
            global_gap = np.inf
        print(f"Maximum relaxation gap (|z - p*q|) among PT+Bike arcs: {global_gap:.6f}")

        # Save current iteration data.
        solutions_list.append({
            "iteration": iteration,
            "od_bounds": od_bounds,
            "global_gap": global_gap,
            "solution": solution,
            "solution_dict": solution_dict,
            "solution_file": solution_file
        })

        if global_gap <= tol:
            print("Relaxation gap is within tolerance. Accepting current solution.")
            break
        
        # Update od_bounds for next iteration.
        od_bounds = new_od_bounds
        iteration += 1

    if not solutions_list:
        return None, None, None
    # Select best solution based on criteria: e.g., lowest global gap, then highest objective.
    best_entry = sorted(solutions_list, key=lambda x: (x["global_gap"], -x["solution_dict"].get("Objective Value", -np.inf)))[0]
    print(f"\nSelected best iteration: {best_entry['iteration']} with global gap = {best_entry['global_gap']:.6f} and objective = {best_entry['solution_dict'].get('Objective Value', None)}")
    
    return best_entry["solution"], best_entry["solution_dict"], best_entry["solution_file"]

# Example usage:
data_folder = r"C:\Users\mkoseogl\Desktop\Instances\Instance_Generator\Generated_Instances\json\Smol"
all_files = os.listdir(data_folder)
instance_paths = [os.path.join(data_folder, file) for file in all_files if file.endswith(".json") and not file.endswith("_solution.json")]
save_folder = r"C:\Users\mkoseogl\Desktop\Instances\Instance_Generator\Generated_Instances\json\experiment_tighter_bounds_iter_od"

solution_paths_tight_iter_od = []
for instance_file in instance_paths:
    print(f"Solving instance: {instance_file}")
    _, _, solution_path = iterative_tightening(
        instance_file,
        save_folder,
        initial_Pbar=0.5,
        initial_M=4,
        ops=min,
        tol=1e-6,
        max_iter=20,
        verbose=False
    )
    solution_paths_tight_iter_od.append(solution_path)

print("\nGenerated Solutions:\n", solution_paths_tight_iter_od)


Solving instance: C:\Users\mkoseogl\Desktop\Instances\Instance_Generator\Generated_Instances\json\Smol\random_instance_42_10_7_90.json

Iteration 0: Solving with OD-specific price bounds:
Set parameter Username
Academic license - for non-commercial use only - expires 2025-05-16
Set parameter NonConvex to value 2
Set parameter Presolve to value 0
Set parameter LogFile to value "C:\Users\mkoseogl\Desktop\Instances\Instance_Generator\Generated_Instances\json\experiment_tighter_bounds_iter_od\gurobi_outputs\gurobi_output_random_instance_42_10_7_90.txt"
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22000.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 3662 rows, 2428 columns and 9968 nonzeros
Model fingerprint: 0x1e48e0cc
Variable types: 1866 continuous, 562 integer (562 binary)
Coefficient

In [3]:
# =====================================================
# 4. Display Dataframe for Solved Instances
# =====================================================

import pandas as pd
import numpy as np

solution_paths_tight_iter =    ['C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_10_7_90_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_13_8_156_3_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_16_13_240_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_18_17_306_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_52_48_2652_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_72_70_5112_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_74_70_5402_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_82_77_6642_1_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_85_83_7140_0_solution.json', 'C:\\Users\\mkoseogl\\Desktop\\Instances\\Instance_Generator\\Generated_Instances\\json\\experiment_tighter_bounds_iter_od\\random_instance_42_8_7_56_0_solution.json']


# Store results from all solutions
all_summaries_tight_iter = []

# Loop through all generated solutions
for solution_file in solution_paths_tight_iter:
    # Ensure the solution file exists
    if os.path.exists(solution_file):
        with open(solution_file, "r") as f:
            data = json.load(f)
        solution_time = round(data["Solution Time (s)"], 2)  # Extract solution time
        Obj_Value= round(data["Objective Value"], 2)         # Extract objective value
        Gap = round(data["Gap (%)"], 2)              # Extract gap value
        df = pd.DataFrame(data["instances"])

        # Extract instance name
        instance_name = os.path.basename(solution_file).replace("_solution.json", "")

        # Compute key statistics
        num_od_pairs = df["OD"].nunique()  # Count unique OD pairs
        total_demand = round(df["Flow (q)"].sum(), 2)  # Total flow (demand)
        total_flow_car = round(df[df["Type"] == "Car"]["Flow (q)"].sum(), 2)  # Total flow for Car
        total_flow_pt_only = round(df[df["Type"] == "PT-Only"]["Flow (q)"].sum(), 2)  # Total flow for PT-Only
        total_flow_pt_bike = round(df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)]["Flow (q)"].sum(), 2)  # Total flow for PT+Bike

        # Compute Average Bike Price for PT+Bike where Flow ≥ 1
        avg_bike_price_pt_bike = df[(df["Type"] == "PT+Bike")& (df["Flow (q)"] >= 1)]["Bike Price"].mean()
        avg_bike_price_pt_bike_total = df[(df["Type"] == "PT+Bike") & (df["nu"] == 1)]["Bike Price"].mean()
        avg_num_transfers_pt_bike = df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)]["Transfers"].mean()
        avg_num_transfers_pt_only = df[(df["Type"] == "PT-Only") & (df["Flow (q)"] >= 1)]["Transfers"].mean()
        
        # **NEW METRICS**
        total_pt_only_arcs = df[df["Type"] == "PT-Only"].shape[0]  # Total PT-Only arcs
        total_pt_bike_arcs = df[df["Type"] == "PT+Bike"].shape[0]  # Total PT+Bike arcs
        used_pt_only_arcs = df[(df["Type"] == "PT-Only") & (df["Flow (q)"] >= 1)].shape[0]  # PT-Only arcs with flow > 0
        used_pt_bike_arcs = df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)].shape[0]  # PT+Bike arcs with flow > 0
        relax_difs = abs(df[(df["Type"] == "PT+Bike")]["Relaxation Difference"]).max()
        z_values = round(df[(df["Type"] == "PT+Bike")]["z value"].max(), 2)
        max_z_row = df[(df["Type"] == "PT+Bike") & (df["z value"] == z_values)]
        price_max= df[(df["Type"] == "PT+Bike")]["Bike Price"].max()
        price_min = df[(df["Type"] == "PT+Bike") & (df["Bike Price"] > 0)]["Bike Price"].min()
        # Append to summary list
        all_summaries_tight_iter.append({
            "Instance Name": instance_name,
            "Solution Time (s)": solution_time,
            "Obj Value": Obj_Value,
            "Gap (%)": Gap,
            "Relaxation Difference": relax_difs,
            "z_max": z_values,
            "p_max": price_max,
            #"p_min": price_min,
            #"Number of OD Pairs": num_od_pairs,
            #"Total Demand": total_demand,
            "Total Flow (Car)": total_flow_car,
            "Total Flow (PT Only)": total_flow_pt_only,
            "Total Flow (Multimodal)": total_flow_pt_bike,
            "Avg Bike Price (for all offered arcs)": avg_bike_price_pt_bike_total,
            "Avg Bike Price (Multimodal, Flow ≥ 1)": avg_bike_price_pt_bike,
            "Avg Number of Transfers (PT Only)": avg_num_transfers_pt_only,
            "Avg Number of Transfers (Multimodal)": avg_num_transfers_pt_bike,
            "Total PT Only Arcs": total_pt_only_arcs,
            "Used PT Only Arcs (Flow >= 1)": used_pt_only_arcs,
            "Total Multimodal Arcs": total_pt_bike_arcs,
            "Used Multimodal Arcs (Flow >= 1)": used_pt_bike_arcs
        })

# Convert to DataFrame and Print
summary_df_tight_iter = pd.DataFrame(all_summaries_tight_iter)
# 1) Compute usage ratios
summary_df_tight_iter["PT Only Arcs Usage Ratio"] = (
    summary_df_tight_iter["Used PT Only Arcs (Flow >= 1)"] / (summary_df_tight_iter["Total PT Only Arcs"] + summary_df_tight_iter["Total Multimodal Arcs"])
)
summary_df_tight_iter["Multimodal Arcs Usage Ratio"] = (
    summary_df_tight_iter["Used Multimodal Arcs (Flow >= 1)"] / (summary_df_tight_iter["Total PT Only Arcs"] + summary_df_tight_iter["Total Multimodal Arcs"])
)

# Melt into long form
usage_cols = ["PT Only Arcs Usage Ratio", "Multimodal Arcs Usage Ratio"]
df_usage = summary_df_tight_iter.melt(
    id_vars=["Instance Name"],
    value_vars=usage_cols,
    var_name="Arc Type",
    value_name="Usage Ratio"
)
display(summary_df_tight_iter.style.set_caption("Summary of Solved Instances with Tighter Flow Bounds with Iterated Price Bounds"))
#display(summary_df_tight.style.set_caption("Summary of Solved Instances with Tighter Flow Bounds"))





,Instance Name,Solution Time (s),Obj Value,Gap (%),Relaxation Difference,z_max,p_max,Total Flow (Car),Total Flow (PT Only),Total Flow (Multimodal),Avg Bike Price (for all offered arcs),"Avg Bike Price (Multimodal, Flow ≥ 1)",Avg Number of Transfers (PT Only),Avg Number of Transfers (Multimodal),Total PT Only Arcs,Used PT Only Arcs (Flow >= 1),Total Multimodal Arcs,Used Multimodal Arcs (Flow >= 1),PT Only Arcs Usage Ratio,Multimodal Arcs Usage Ratio
0,random_instance_42_10_7_90_0,0.080000,0.000000,0.000000,0.000000,0.000000,4.000000,1424.000000,12983.000000,0.000000,2.434859,nan,0.914634,nan,278,82,284,0,0.145907,0.000000
1,random_instance_42_13_8_156_3,0.090000,0.000000,0.000000,0.000000,0.000000,3.825000,1758.000000,25610.000000,0.000000,1.600873,nan,0.805556,nan,500,144,410,0,0.158242,0.000000
2,random_instance_42_16_13_240_0,0.380000,152.600000,0.000000,4.214980,5.270000,4.000000,2539.000000,40453.930000,19.070000,2.393226,0.633333,0.816964,1.333333,731,224,725,3,0.153846,0.002060
3,random_instance_42_18_17_306_0,0.590000,203.710000,0.000000,15.000000,4.740000,4.000000,3677.000000,48660.540000,25.460000,2.438056,1.294235,0.853147,1.750000,933,286,948,4,0.152047,0.002127
4,random_instance_42_52_48_2652_0,2.310000,13584.210000,0.000000,33.612497,20.000000,4.000000,37304.290000,423197.680000,1696.800000,2.132406,1.048045,0.813934,1.318841,8094,2440,7888,69,0.152672,0.004317
5,random_instance_42_72_70_5112_0,6.720000,19910.440000,0.000000,33.335664,19.500000,4.000000,72383.150000,820888.050000,2488.800000,0.700788,0.678334,0.814799,1.346535,15211,4703,15183,101,0.154734,0.003323
6,random_instance_42_74_70_5402_0,8.350000,23564.240000,0.000000,33.201515,20.000000,4.000000,75612.870000,862548.600000,2944.710000,0.735139,0.657425,0.820652,1.354545,16297,4968,16170,110,0.153017,0.003388
7,random_instance_42_82_77_6642_1,11.800000,13494.760000,0.000000,25.420651,111.040000,4.000000,87734.620000,1075967.200000,1670.180000,2.572453,0.730746,0.810471,1.396552,19842,6131,20000,58,0.153883,0.001456
8,random_instance_42_85_83_7140_0,12.350000,26316.840000,0.000000,23.321555,19.280000,4.000000,97414.030000,1151603.370000,3289.000000,0.667781,0.569363,0.812946,1.346457,21404,6581,21377,127,0.153830,0.002969
9,random_instance_42_8_7_56_0,0.070000,0.000000,0.000000,0.000000,0.000000,0.500000,381.000000,8683.000000,0.000000,0.500000,nan,0.811321,nan,162,53,167,0,0.161094,0.000000
